# Qwen2.5-1.5B-Instruct GRPO Training on DevOpsEnv

**Selected policy:** Group Relative Policy Optimization (GRPO) with LoRA.

Why GRPO for this system:
- Multi-agent IC-driven decision making → policy is the LLM that emits JSON `next_action`+`target_agent`.
- Delayed reward (terminal `success_reward` ±2.0) + dense shaping → episode-level return is the cleanest learning signal.
- Long-horizon (≤15 steps) with high variance → group-relative advantages (G rollouts per scenario) cut variance without a critic.
- 1.5B fits Colab T4/A100 with LoRA + bf16/fp16; no value head needed.

**Strict rule:** `inference.py`, `env.py`, `models.py`, `multi_agent.py`, rewards, dataset are NOT modified. We import their prompt/parsing helpers read-only and drive the env via `WarRoom`.

## 1. Setup

In [ ]:
!pip -q install "transformers>=4.44" "trl>=0.10" "accelerate>=0.33" "peft>=0.12" \
    "datasets>=2.20" "bitsandbytes>=0.43" "pydantic>=2.6" "python-dotenv" "openai" \
    "matplotlib" "sentencepiece" "safetensors"

In [ ]:
import os, sys
REPO_DIR = '/content/Hackathon'
if not os.path.isdir(REPO_DIR):
    os.makedirs(REPO_DIR, exist_ok=True)
    print('Upload Hackathon repo files (env.py, inference.py, models.py, multi_agent.py, tasks/cascade.json) into', REPO_DIR)
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        target = os.path.join(REPO_DIR, name)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with open(target, 'wb') as f:
            f.write(data)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo dir:', REPO_DIR)
print(os.listdir(REPO_DIR))

In [ ]:
os.environ.setdefault('HF_TOKEN', 'dummy')
os.environ.setdefault('API_BASE_URL', 'http://localhost')
os.environ.setdefault('MODEL_NAME', 'Qwen/Qwen2.5-1.5B-Instruct')

## 2. Imports — read-only use of existing system

In [ ]:
import json, copy, math, random, time
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
import matplotlib.pyplot as plt

import inference as inf
from env import DevOpsEnv, AGENT_DOMAIN_MAP, EXECUTION_AGENTS
from multi_agent import WarRoom
from models import Action, Reward

MAX_STEPS = inf.MAX_STEPS
print('MAX_STEPS =', MAX_STEPS, '| agents =', EXECUTION_AGENTS)

## 3. Load Qwen 1.5B Instruct + LoRA policy and frozen reference

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
base.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
policy = get_peft_model(base, lora_cfg)
policy.print_trainable_parameters()

ref_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

## 4. Local generation hook — replace `inference.call_llm` with our policy

In [ ]:
GEN_TEMP = 0.7
GEN_TOP_P = 0.9
MAX_NEW = 256

TRAJECTORY_BUFFER: List[Dict[str, Any]] = []
_CURRENT_TRAJ: List[Dict[str, Any]] = []
_RECORD_PLANNER_ONLY = True  # train only on planner decisions (delayed-reward signal)

def _is_planner_prompt(prompt: str) -> bool:
    return 'You are the Incident Commander' in prompt

def _generate(prompt: str, max_new_tokens: int, sample: bool):
    msgs = [{'role':'user','content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=4096).to(DEVICE)
    prompt_len = enc.input_ids.shape[1]
    with torch.no_grad():
        out = policy.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            temperature=GEN_TEMP if sample else 1.0,
            top_p=GEN_TOP_P if sample else 1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    full = out[0]
    completion_ids = full[prompt_len:]
    completion = tokenizer.decode(completion_ids, skip_special_tokens=True)
    return completion, enc.input_ids[0].detach().cpu(), completion_ids.detach().cpu()

def call_llm_local(prompt: str, max_tokens: int = 250) -> str:
    sample = _is_planner_prompt(prompt)  # only sample where we train; obs is greedy
    completion, prompt_ids, comp_ids = _generate(prompt, max_tokens, sample=sample)
    if _RECORD_PLANNER_ONLY and sample and len(comp_ids) > 0:
        _CURRENT_TRAJ.append({
            'prompt_ids': prompt_ids,
            'completion_ids': comp_ids,
        })
    return completion

# Monkey-patch ONLY the network call. inference.py logic, prompts, parsing remain untouched.
inf.call_llm = call_llm_local
print('Patched inference.call_llm with local Qwen generation.')

## 5. Data collection — rollouts via existing `WarRoom` + `inference._run_episode_core`

In [ ]:
def rollout_one_episode(scenario_idx: int, seed: int) -> Dict[str, Any]:
    global _CURRENT_TRAJ
    _CURRENT_TRAJ = []
    room = WarRoom(seed=seed, max_steps=MAX_STEPS)
    room.env.scenario_index = scenario_idx
    room.reset()
    rewards = inf._run_episode_core(room)
    total = room.get_total_reward()
    progress = room.get_progress()
    success = bool(room.is_done() and total > 0)
    steps = list(_CURRENT_TRAJ)
    _CURRENT_TRAJ = []
    return {
        'scenario_idx': scenario_idx,
        'steps': steps,
        'total_reward': float(total),
        'progress': float(progress),
        'success': success,
        'num_env_steps': room.step_count,
        'step_rewards': rewards,
    }

# quick smoke test (1 episode, will run actual env + Qwen)
_demo = rollout_one_episode(0, seed=123)
print({k: _demo[k] for k in ['scenario_idx','total_reward','progress','success','num_env_steps']}, 'planner_steps:', len(_demo['steps']))

## 6. GRPO training loop

For each scenario we sample **G** trajectories. Episode return $R_i$ is the env's total reward. Group-relative advantage:
$$A_i = \frac{R_i - \mathrm{mean}(R)}{\mathrm{std}(R) + \epsilon}$$
Each planner-step generation in trajectory $i$ inherits $A_i$ (credit-assignment via group baseline; handles delayed reward). Loss combines a clipped policy-gradient term over completion tokens with a KL penalty toward the frozen reference model.

In [ ]:
def logprobs_for(model, prompt_ids: torch.Tensor, completion_ids: torch.Tensor) -> torch.Tensor:
    full = torch.cat([prompt_ids, completion_ids], dim=0).unsqueeze(0).to(DEVICE)
    attn = torch.ones_like(full)
    out = model(input_ids=full, attention_mask=attn)
    logits = out.logits[0, :-1, :]
    targets = full[0, 1:]
    logp = F.log_softmax(logits.float(), dim=-1)
    tok_lp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    comp_len = completion_ids.shape[0]
    return tok_lp[-comp_len:]

def grpo_step(group: List[Dict[str, Any]], optimizer, kl_coef=0.05, clip_eps=0.2, max_tokens_per_step=256):
    returns = np.array([t['total_reward'] for t in group], dtype=np.float32)
    mean, std = returns.mean(), returns.std() + 1e-6
    advs = (returns - mean) / std
    optimizer.zero_grad()
    total_loss = torch.tensor(0.0, device=DEVICE)
    n_terms = 0
    for traj, A in zip(group, advs):
        if not traj['steps']:
            continue
        A_t = torch.tensor(float(A), device=DEVICE)
        for st in traj['steps']:
            p_ids = st['prompt_ids']
            c_ids = st['completion_ids'][:max_tokens_per_step]
            if c_ids.numel() == 0:
                continue
            new_lp = logprobs_for(policy, p_ids, c_ids)
            with torch.no_grad():
                old_lp = logprobs_for(ref_model, p_ids, c_ids)
            ratio = torch.exp(new_lp - old_lp.detach())
            unclipped = ratio * A_t
            clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * A_t
            pg = -torch.min(unclipped, clipped).mean()
            kl = (new_lp - old_lp.detach()).mean()  # approx KL
            total_loss = total_loss + pg + kl_coef * kl
            n_terms += 1
    if n_terms == 0:
        return None
    loss = total_loss / n_terms
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    return float(loss.detach().cpu()), float(returns.mean()), float(returns.std())

In [ ]:
NUM_SCENARIOS = len(DevOpsEnv._DATA_CACHE) if DevOpsEnv._DATA_CACHE else None
if NUM_SCENARIOS is None:
    _tmp = DevOpsEnv(); _tmp.reset(); NUM_SCENARIOS = len(DevOpsEnv._DATA_CACHE)
print('scenarios available:', NUM_SCENARIOS)

GROUP_SIZE = 4
EPOCHS = 3
LR = 1e-5

trainable = [p for p in policy.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR)

history = {'episode_reward': [], 'group_mean': [], 'group_std': [], 'loss': [], 'success': []}

step_idx = 0
for epoch in range(EPOCHS):
    scenario_order = list(range(NUM_SCENARIOS))
    random.Random(epoch).shuffle(scenario_order)
    for sid in scenario_order:
        group = []
        for g in range(GROUP_SIZE):
            seed = 1000 * epoch + 10 * sid + g
            traj = rollout_one_episode(sid, seed=seed)
            group.append(traj)
            history['episode_reward'].append(traj['total_reward'])
            history['success'].append(int(traj['success']))
        out = grpo_step(group, optimizer)
        if out is None:
            print(f'[ep{epoch} sid{sid}] no trainable steps, skipping')
            continue
        loss_val, gm, gs = out
        history['loss'].append(loss_val); history['group_mean'].append(gm); history['group_std'].append(gs)
        step_idx += 1
        print(f'[ep{epoch} sid{sid}] loss={loss_val:.4f} group_mean_R={gm:+.3f} std={gs:.3f}')

policy.save_pretrained('/content/qwen15b_grpo_lora')
print('Saved LoRA adapters to /content/qwen15b_grpo_lora')

## 7. Evaluation

In [ ]:
def evaluate(num_runs_per_scenario=2, greedy=True):
    global GEN_TEMP
    saved = GEN_TEMP
    GEN_TEMP = 0.0 if greedy else saved
    results = []
    for sid in range(NUM_SCENARIOS):
        for r in range(num_runs_per_scenario):
            t = rollout_one_episode(sid, seed=99991 + 7 * sid + r)
            results.append(t)
            print(f'eval sid={sid} run={r} reward={t["total_reward"]:+.3f} success={t["success"]} steps={t["num_env_steps"]} progress={t["progress"]:.0%}')
    GEN_TEMP = saved
    rewards = [t['total_reward'] for t in results]
    succs = [t['success'] for t in results]
    steps = [t['num_env_steps'] for t in results]
    print(f'\n=== EVAL SUMMARY ===\n mean_reward={np.mean(rewards):+.3f}  success_rate={np.mean(succs):.1%}  mean_steps={np.mean(steps):.2f}')
    return results

eval_results = evaluate(num_runs_per_scenario=2, greedy=True)

## 8. Visualization

In [ ]:
ep_r = np.array(history['episode_reward'])
succ = np.array(history['success'])
win = max(1, len(ep_r)//20)
smoothed = np.convolve(ep_r, np.ones(win)/win, mode='valid') if len(ep_r) >= win else ep_r

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ep_r, alpha=0.4, label='per-episode'); axes[0].plot(np.arange(len(smoothed))+win-1, smoothed, label=f'rolling{win}')
axes[0].set_title('Episode reward'); axes[0].set_xlabel('episode'); axes[0].grid(True); axes[0].legend()

if history['loss']:
    axes[1].plot(history['loss']); axes[1].set_title('GRPO loss'); axes[1].set_xlabel('grad step'); axes[1].grid(True)
if history['group_mean']:
    axes[2].plot(history['group_mean'], label='group mean R')
    axes[2].fill_between(range(len(history['group_mean'])),
                         np.array(history['group_mean']) - np.array(history['group_std']),
                         np.array(history['group_mean']) + np.array(history['group_std']), alpha=0.2)
    axes[2].set_title('Group mean ± std'); axes[2].set_xlabel('update'); axes[2].grid(True); axes[2].legend()
plt.tight_layout(); plt.show()

if len(succ) > 0:
    sw = max(1, len(succ)//10)
    sr = np.convolve(succ.astype(float), np.ones(sw)/sw, mode='valid')
    plt.figure(figsize=(8,3)); plt.plot(sr); plt.title(f'Rolling success rate (win={sw})'); plt.grid(True); plt.show()